# Занятие 6. Частотная область и пирамиды

Курс «Введение в компьютерное зрение» · Innopolis University · Fall 2026
Лекция-опора: L6 «Частотная область и пирамиды» · 90 минут в классе (последние 20 — клиника ДЗ) · ведёт ассистент

---

Сегодня проверяем **теорему о свёртке** числом: свёртка в пространстве — умножение спектров, и цена её не зависит от размера ядра (вот куда делась ступенька `filter2D` из занятия 5). Потом находим в спектре **периодическую помеху** — две точки в тысячи раз выше фона — и вырезаем их notch-фильтром. В третьей части строим **свою пирамиду Гаусса и Лапласа** без `cv2.pyrDown` и проверяем обратимость. В четвёртой — **клеим два кадра по полосам частот**: широкий переход для яркости и узкий для деталей одновременно.

Заготовка начинается на слайдах 11 и 14 лекции («Как читать спектр» и «Теорема о свёртке») и на демо 1.3.

| Минуты | Что происходит |
|--------|----------------|
| 0–12 | Ассистент разбирает опорный пример: спектр и «фаза важнее амплитуды» |
| 12–58 | Вы делаете TODO 1–4 |
| 58–67 | Разбор решения |
| 67–87 | Клиника: ДЗ 1 — финальные вопросы; ДЗ 2 «Сканер документов» — что из сегодняшнего занятия туда идёт |
| 87–90 | Зачёт |

Ничего сдавать не нужно: зачёт ставится в классе по факту работы (2 % итоговой оценки).
⭐ — необязательная звёздочка. Решение публикуется в репозитории сразу после занятия.
**Занятие 7 — Quiz 2 по блоку II (L4–L6).**

## 0. Проверка окружения

Если ячейка ругается — зовите ассистента сразу, не тратьте время занятия.

In [ ]:
REPO_URL = "https://github.com/afanasyspb/iu-intro-cv.git"     # адрес репозитория курса (для Colab)

import sys, subprocess
try:
    import cvcourse
except ImportError:
    if "google.colab" in sys.modules:      # Colab: пакет курса ставится один раз за сессию
        subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "opencv-contrib-python==4.14.0.94", "git+" + REPO_URL], check=True)
        import cvcourse
    else:
        raise ImportError("пакет курса не установлен: из корня репозитория выполните "
                          "pip install -r requirements.txt (docs/setup-guide.md)")

import os, time, timeit
import cv2, numpy as np
import matplotlib.pyplot as plt
from cvcourse import io as cio, viz, metrics

print("OpenCV", cv2.__version__, "| NumPy", np.__version__, "| Colab:", cvcourse.IN_COLAB)
assert cv2.__version__.startswith("4.14"), "курс собран на OpenCV 4.14.0.94"

## 1. Берём данные

Всё лежит в репозитории — скачивать нечего. Фото здания ИУ `lectures/L06-frequency/demo/iu-building.jpg`
(1920 × 1279) читается в сером; кроп лестницы 256 × 256 тот же, что в занятии 5.

Помеха порождается здесь же: косая синусоидальная решётка, прибавленная к кадру (модель слайдов 5 и 20:
наводка моторов, мерцание ламп, растр). У неё три свойства, ради которых всё затевалось:

- она **периодическая** — в спектре это пара ярких точек;
- она **прибавлена** — спектр помехи лежит отдельно от спектра кадра (линейность);
- её частота **не кратна размеру кадра** — пик расплывается в «крест», и провал нужен не в одну точку.

Для склейки (TODO 4) пара кадров строится из того же фото: правая половина сдвинута на 3 px и темнее
на 24 кода — модель слайдов 37–38.

In [ ]:
LECTURE_DIR = "lectures/L06-frequency/demo"
CANDIDATE_DIRS = ["../../" + LECTURE_DIR,                     # публичный репозиторий: labs/lab06-frequency/
                  "../../../iu-intro-cv/" + LECTURE_DIR,      # ноутбук открыт из iu-intro-cv-ta
                  "data"]                                     # уже скачанное
RAW_URL = "https://raw.githubusercontent.com/afanasyspb/iu-intro-cv/main/" + LECTURE_DIR + "/"
FILE = "iu-building.jpg"
CROP = (slice(880, 1136), slice(1300, 1556))                  # лестница с перилами, 256 × 256 (то же, что в занятии 5)


def synthetic_photo(h=1279, w=1920):
    """Запас без сети: плавный фон, круги, «ступени» и «прутья» на месте кропа. Числа будут другие."""
    yy, xx = np.mgrid[:h, :w].astype(np.float32)
    img = 120 + 60 * np.sin(xx / 97.0) * np.cos(yy / 71.0) + 0.03 * xx
    for cx, cy, r in ((500, 400, 220), (1400, 900, 300), (1000, 250, 120)):
        img[np.hypot(xx - cx, yy - cy) < r] += 50
    img[880:1136:16, 1300:1556] -= 70
    img[880:1136, 1300:1556:32] -= 50
    return np.uint8(np.clip(img, 0, 255))


def load_photo():
    """Фото лекции: (серое, откуда взято)."""
    for d in CANDIDATE_DIRS:
        p = os.path.join(d, FILE)
        if os.path.exists(p):
            return cio.imread(p, "gray"), d
    try:                                                      # Colab: репозитория рядом нет — качаем
        import urllib.request
        os.makedirs("data", exist_ok=True)
        p = os.path.join("data", FILE)
        urllib.request.urlretrieve(RAW_URL + FILE, p)
        return cio.imread(p, "gray"), RAW_URL
    except Exception as e:
        print("файл лекции не скачался (%s) — синтетическая сцена" % type(e).__name__)
        return synthetic_photo(), "синтетика"


def add_periodic_noise(img, amp=18.0):
    """Косая решётка, прибавленная к кадру: пара точек в спектре, частота не кратна размеру (слайды 5, 20)."""
    h, w = img.shape
    yy, xx = np.mgrid[:h, :w].astype(np.float64)
    pat = amp * np.sin(2 * np.pi * (xx * 65 / 1920 + yy * 129 / 1280))   # 1280 — оптимальный размер ДПФ высоты 1279
    return np.clip(img.astype(np.float64) + pat, 0, 255).astype(np.uint8)


photo, SRC = load_photo()
assert photo.shape == (1279, 1920) and photo.dtype == np.uint8, "ожидался серый кадр 1920 × 1279, uint8"
crop = photo[CROP]
noisy = add_periodic_noise(photo)
db_noisy = metrics.psnr(photo, noisy)
print("источник:", SRC, "· кадр", photo.shape, photo.dtype, "· PSNR кадра с помехой %.1f дБ" % db_noisy)
viz.grid({"кроп лестницы": crop, "помеха": noisy[CROP]}, cols=2, size=3.4, suptitle="кадр и периодическая помеха")

## 2. Опорный пример — разбирает ассистент

Здесь всё написано, задача — **понять каждую строку**.

**Спектр — тот же кадр в другом базисе** (слайд 9): `F(u, v)` — комплексное число; `|F|` — сколько в кадре решётки с `u` периодами по ширине и `v` по высоте; `arg F` — где лежат её гребни. После `fftshift` нулевая частота в центре; `F(0, 0)` — сумма всех пикселей, 3.3·10⁸. Смотрим `log(1 + |F|)` — динамический диапазон амплитуды девять порядков.

In [ ]:
F0 = np.fft.fftshift(np.fft.fft2(np.float64(photo)))
A0 = np.log1p(np.abs(F0))
cy, cx = photo.shape[0] // 2, photo.shape[1] // 2
print("F(0,0) — сумма всех пикселей: %.3g" % np.abs(F0[cy, cx]))
print("80%% энергии — в 0.03%% коэффициентов у центра (слайд 11)")
viz.grid({"лог-амплитуда спектра": A0 / A0.max()}, cols=1, size=5)
# лучи в спектре — следы прямых линий кадра, каждый перпендикулярен своим линиям:
# вертикали фасада дают полосу вдоль оси u, ступени и перила — лучи поперёк них

### Сдвиг меняет только фазу

Слайд 13. Сдвинули кадр — амплитуда та же до 10⁻¹⁶, изменилась только фаза. На этом стоит фазовая корреляция: `cv2.phaseCorrelate` находит сдвиг между кадрами за 10 мс без единой точки интереса.

In [ ]:
shifted = np.roll(photo, (3, 2), axis=(0, 1))                       # сдвиг на (3, 2) px — циклический, для чистоты фазы
Fs = np.fft.fft2(np.float64(shifted))
Fu = np.fft.fft2(np.float64(photo))
print("амплитуда сдвинутого кадра отличается на %.1e — фаза хранит место, амплитуда нет" % np.abs(np.abs(Fs) - np.abs(Fu)).max())
print("преобразование обратимо: ifft2(fft2(I)) — ошибка %.1e" % np.abs(np.fft.ifft2(Fu).real - photo).max())

### Вопрос, на который отвечаем вслух

Кадр цветной, форма `H × W × 3`. Какие оси возьмёт `np.fft.fft2` по умолчанию, что получится в `F[0, 0]` и как правильно?

---

## TODO 1 — свёртка через спектр (≈ 12 минут)

Слайд 14. **Теорема о свёртке:** свёртка в пространстве — умножение спектров, `I ∗ h ⇔ F · H`.

Напишите `conv_dft(img, kernel)` — свёртку серого кадра с ядром через ДПФ. Результат — `float64` того же размера; внутри кадра он должен совпадать с `cv2.filter2D` с перевёрнутым ядром и границей `BORDER_CONSTANT` до 10⁻⁹.

Скелет — шесть шагов:

1. размер ДПФ — `cv2.getOptimalDFTSize(h + kh - 1)` по каждой оси: это **линейная** свёртка, а не циклическая, и на оптимальном размере (2ᵃ3ᵇ5ᶜ) FFT быстрее;
2. `fft2` кадра, затем `fftshift` — нулевая частота в центр;
3. `fft2` ядра того же размера, `fftshift`, и **`np.roll(..., (kh // 2, kw // 2))`** — нуль ядра (его опорная точка) должен совпасть с центром ДПФ-массива, иначе фаза умножения произвольная;
4. поэлементное умножение `F * K`;
5. `ifft2` и **`.real`** — мнимая часть есть числовой шум порядка 10⁻¹³; `np.abs` здесь брать нельзя: он сделает шум положительным;
6. вырезать `[:h, :w]` — левый верх результата.

> **Подсказка.** У края кадра результат **не обязан** совпадать с `filter2D` с `REFLECT_101`: ДПФ считает кадр периодическим, окно «заворачивается» через край. Ассерты проверяют только внутренность кадра — это не баг, это граница применимости (слайд 14).

In [ ]:
def conv_dft(img, kernel):
    """Свёртка серого кадра с ядром через ДПФ → float64 того же размера; внутри кадра совпадает с filter2D(flip, BORDER_CONSTANT)."""
    img, kernel = np.asarray(img, np.float64), np.asarray(kernel, np.float64)
    h, w = img.shape; kh, kw = kernel.shape
    dh = cv2.getOptimalDFTSize(h + kh - 1)
    dw = cv2.getOptimalDFTSize(w + kw - 1)
    kpad = np.zeros((dh, dw), np.float64)
    kpad[:kh, :kw] = kernel
    kpad = np.roll(kpad, (-(kh // 2), -(kw // 2)), axis=(0, 1))
    F = np.fft.fftshift(np.fft.fft2(img, s=(dh, dw)))
    K = np.fft.fftshift(np.fft.fft2(kpad))
    return np.fft.ifft2(np.fft.ifftshift(F * K)).real[:h, :w]


K_TEST = np.float64([[1, 2, 0], [0, 3, -1], [-2, 0, 1]])                      # нарочно несимметричное (занятие 5)
flt_c = lambda im, kern: cv2.filter2D(np.float64(im), cv2.CV_64F, np.float64(kern), borderType=cv2.BORDER_CONSTANT)
ref_c = flt_c(crop, cv2.flip(K_TEST, -1))                                     # свёртка = перевёрнутое ядро (занятие 5)
mine = conv_dft(crop, K_TEST)
assert isinstance(mine, np.ndarray) and mine.shape == crop.shape and mine.dtype == np.float64, "результат — float64 того же размера, что кадр"
delta = np.zeros((3, 3)); delta[1, 1] = 1                                     # δ-ядро: свёртка ничего не меняет
assert np.abs(conv_dft(crop, delta) - crop).max() < 1e-9, "δ-ядро должно возвращать кадр: нуль ядра — в центр ДПФ-массива (fftshift + np.roll на половину ядра)"
d_in = np.abs(mine - ref_c)[8:-8, 8:-8].max()
assert d_in < 1e-9, "внутри кадра расхождение с filter2D %.3g: порядок — ДПФ размера h + kh − 1, fftshift обоих, .real после ifft2" % d_in
K_RECT = np.arange(21, dtype=np.float64).reshape(3, 7) - 8                    # ядро 3 × 7: не перепутаны ли оси размера ДПФ?
assert np.abs(conv_dft(crop, K_RECT) - flt_c(crop, cv2.flip(K_RECT, -1)))[8:-8, 8:-8].max() < 1e-9, "ядро 3 × 7: размер ДПФ по высоте — от kh, по ширине — от kw"
shift = np.zeros((5, 5)); shift[2, 4] = 1                                     # ядро опорного примера занятия 5
assert np.abs(conv_dft(crop, shift)[:, 2:] - crop[:, :-2]).max() < 1e-9, "свёртка с ядром «единица справа на 2» сдвигает кадр вправо на 2"
print("TODO 1 ✔ · внутри кадра против filter2D: %.1e · у края расхождение %.0f кодов — ДПФ считает кадр периодическим" % (d_in, np.abs(mine - ref_c).max()))

### Теорема о свёртке — числом, на полном кадре

Слайд 14: `filter2D` и `ifft2(F · H)` совпадают до 5·10⁻¹³ внутри кадра; у края расходятся — спектр считает кадр периодическим. Спектр box 9 × 9 обращается в ноль на частотах `k / 9` — в спектре результата на этих местах тёмная сетка.

In [ ]:
box9 = np.ones((9, 9)) / 81.0
sp = conv_dft(photo, box9)
ref = flt_c(photo, cv2.flip(box9, -1))
print("полный кадр, box 9 × 9: внутри %.1e кодов · у края %.0f кодов — окно заворачивается через край" % (np.abs(sp - ref)[16:-16, 16:-16].max(), np.abs(sp - ref).max()))
H9 = np.abs(np.fft.fftshift(np.fft.fft2(box9, s=photo.shape)))
viz.grid({"спектр результата": np.log1p(np.abs(np.fft.fftshift(np.fft.fft2(sp)))),
          "частотная характеристика box 9 × 9": H9 / H9.max()}, cols=2, size=4,
         suptitle="нули спектра box — тёмная сетка на частотах k/9 (слайд 14)")

### Цена спектра: почему `filter2D` после `k ≈ 11` почти не растёт

Загадка ступеньки из занятия 5. Прямая свёртка через ДПФ стоит **одинаково** при любом размере ядра: две прямых и одна обратная свёртка, ядро входит бесплатно. Вот почему `filter2D` делает ступеньку и выходит на полку — он сам переходит на спектр.

In [ ]:
ms = lambda fn, repeat=3: 1e3 * min(timeit.repeat(fn, number=1, repeat=repeat))       # лучшее из нескольких прогонов, мс
print("кроп 256 × 256:")
for k in (5, 31, 101):
    kern = np.random.default_rng(7).normal(0, 1, (k, k)) / k                          # полное несепарабельное ядро
    print("   k = %3d: conv_dft %6.1f мс · filter2D %6.2f мс" % (k, ms(lambda: conv_dft(crop, kern), 2), ms(lambda: flt_c(crop, cv2.flip(kern, -1)), 5)))
print("кадр 1920 × 1279, ядро 5 против 101:")
for k in (5, 101):
    kern = np.random.default_rng(7).normal(0, 1, (k, k)) / k
    print("   k = %3d: conv_dft %6.1f мс · время не зависит от размера ядра" % (k, ms(lambda: conv_dft(photo, kern), 2)))

### Вопрос, на который отвечаем вслух

Свёртка через спектр стоит ≈ 50 мс при любом размере ядра, а `sepFilter2D` — 0.7–5 мс на маленьких ядрах. Когда спектр оправдан, а когда — нет? (Ответ — слайд 48: «Спектр или ядро: когда что».)

---

## TODO 2 — найти пик помехи и погасить его (≈ 15 минут)

Слайды 5, 20. Помеха **прибавлена** к кадру и **периодична** — в спектре это пара точек в тысячи раз выше фона, а Гаусс и билатеральный на этой помехе остаются при своих ≈ 24–26 дБ: пространственные фильтры не знают, что помеху можно просто **вычесть из спектра**.

Напишите две функции:

- `find_peak(spec)` — спектр в формате «нуль в углу» (как возвращает `fft2`). Вернуть `(du, dv)` — смещение самого яркого пика **относительно центра**, не считая окрестность нулевой частоты (центр ±32 занулить: `F(0,0)` — сумма пикселей, 3.3·10⁸, это не помеха);
- `notch_reject(img, r=16)` — погасить пик и его **зеркальную пару** гауссовым провалом радиуса `r`: множитель маски `1 − exp(−d² / 2r²)` вокруг каждой из двух точек; вернуться в кадр через `ifft2`, взять `.real` и обрезать в 0–255.

Три ловушки из слайдов 58–59:

1. **забытый `fftshift`** — нулевая частота в углу, маска в центре: ФНЧ превращается в ФВЧ;
2. **`np.abs` вместо `.real`** — модуль переворачивает отрицательные значения;
3. **прямоугольная маска** — идеальный фильтр звенит на 9% скачка и выводит значения за диапазон байта (слайд 18); провал — только Гаусс.

> **Подсказка.** Решётка в кадре — **пара** точек в спектре: пик и `(−du, −dv)` относительно центра. Подавить нужно обе.

In [ ]:
def find_peak(spec):
    """Спектр (нуль в углу) → (du, dv): смещение пика помехи относительно центра; окрестность DC (±32) не считается."""
    amp = np.abs(np.fft.fftshift(np.asarray(spec)))
    h, w = amp.shape
    cy, cx = h // 2, w // 2
    amp = amp.copy()
    amp[max(0, cy - 32):min(h, cy + 33), max(0, cx - 32):min(w, cx + 33)] = 0
    y, x = np.unravel_index(np.argmax(amp), amp.shape)
    return int(y - cy), int(x - cx)


def notch_reject(img, r=16):
    """Погасить пик помехи и зеркальную пару гауссовым провалом радиуса r → float64 в диапазоне 0–255."""
    x = np.asarray(img, np.float64)
    F = np.fft.fftshift(np.fft.fft2(x))
    du, dv = find_peak(np.fft.ifftshift(F))
    h, w = x.shape
    cy, cx = h // 2, w // 2
    yy, xx = np.mgrid[:h, :w]
    d1 = (yy - (cy + du)) ** 2 + (xx - (cx + dv)) ** 2
    d2 = (yy - (cy - du)) ** 2 + (xx - (cx - dv)) ** 2
    mask = (1 - np.exp(-d1 / (2.0 * r * r))) * (1 - np.exp(-d2 / (2.0 * r * r)))
    out = np.fft.ifft2(np.fft.ifftshift(F * mask)).real
    return np.clip(out, 0, 255)


S = np.fft.fft2(np.float64(noisy))
du, dv = find_peak(S)
h, w = noisy.shape; cy0, cx0 = h // 2, w // 2
A_shift = np.abs(np.fft.fftshift(S))
bg = np.median(A_shift)
assert (du, dv) != (0, 0) and abs(du) + abs(dv) > 40, "пик не в центре и не рядом с DC: окрестность ±32 занулена?"
ratio = A_shift[cy0 + du, cx0 + dv] / bg
assert ratio > 1e3, "пик должен быть в тысячи раз выше фона (сейчас ×%.0f)" % ratio
Mchk = (1 - np.exp(-(np.hypot(np.mgrid[0:h, 0:w][0] - (cy0 + du), np.mgrid[0:h, 0:w][1] - (cx0 + dv)) ** 2) / (2 * 16.0 ** 2))) \
       * (1 - np.exp(-(np.hypot(np.mgrid[0:h, 0:w][0] - (cy0 - du), np.mgrid[0:h, 0:w][1] - (cx0 - dv)) ** 2) / (2 * 16.0 ** 2)))
assert np.abs(Mchk - np.roll(np.flip(Mchk), 1, axis=1)).max() < 1e-12, "маска должна быть симметрична относительно центра: гасим пик и зеркальную пару"
res = notch_reject(noisy, 16)
db_notch = metrics.psnr(photo, np.clip(res, 0, 255))
assert res.shape == noisy.shape and res.min() >= 0 and res.max() <= 255, "результат — тот же размер, значения в 0–255 (.real и клип)"
assert db_notch > db_noisy + 5, "notch должен поднять PSNR хотя бы на 5 дБ (сейчас %.1f → %.1f): обе ли точки погашены?" % (db_noisy, db_notch)
print("TODO 2 ✔ · пик на (%+d, %+d), ×%.0f к фону · помеха %.1f дБ → notch r=16: %.1f дБ" % (du, dv, ratio, db_noisy, db_notch))

### Радиус провала: сколько спектра гасить (слайд 19)

Частота помехи не кратна размеру кадра — пик расплылся в «крест». Провал в одну точку не достанет крылья, слишком широкий — начнёт есть кадр. Измеряем PSNR против `r` и сравниваем с пространственными фильтрами.

In [ ]:
print("r, отсчёты   |  2    16    32   128")
vals = [metrics.psnr(photo, notch_reject(noisy, r)) for r in (2, 16, 32, 128)]
print("PSNR, дБ      | " + " ".join("%5.1f" % v for v in vals))
print("для сравнения на этой же помехе:")
print("   Гаусс σ 1.5      %.1f дБ" % metrics.psnr(photo, cv2.GaussianBlur(noisy, (0, 0), 1.5)))
print("   билатеральный 9  %.1f дБ" % metrics.psnr(photo, cv2.bilateralFilter(noisy, 9, 75, 75)))
viz.grid({"помеха": noisy[CROP], "notch r=2": np.clip(notch_reject(noisy, 2), 0, 255).astype(np.uint8)[CROP],
          "notch r=16": np.clip(notch_reject(noisy, 16), 0, 255).astype(np.uint8)[CROP],
          "notch r=128": np.clip(notch_reject(noisy, 128), 0, 255).astype(np.uint8)[CROP]}, cols=4, size=2.8)

### Вопрос, на который отвечаем вслух

На бинарном растре с шагом 6 px notch дал всего 0.2 дБ выигрыша против Гаусса (слайд 21). Почему фильтр, который только что поднял кадр на 12 дБ, здесь почти бесполезен?

---

## TODO 3 — своя пирамида: `pyr_down`, `pyr_up`, Гаусс и Лаплас (≈ 10 минут)

Слайды 39–41. Шаг пирамиды — размыть ядром `[1 4 6 4 1] / 16` по каждой оси и выбросить каждый второй пиксель (`cv2.pyrDown`). Пирамида Лапласа — разности соседних уровней: то, что потерялось при уменьшении; кадр разложен по октавам частот и **обратим**.

Напишите пять функций (**без `cv2.pyrDown` / `cv2.pyrUp`** в `pyr_down` и `pyr_up`):

1. `pyr_down(x)` — два сепарабельных прохода ядром `[1,4,6,4,1]/16` с достройкой `REFLECT_101`, затем каждый второй пиксель. Размер результата — `(H + 1) // 2 × (W + 1) // 2`, как у `cv2.pyrDown`;
2. `pyr_up(x, dstsize)` — вставить нули между отсчётами, применить то же ядро (сепарабельно), умножить на 4 и обрезать до `dstsize = (w, h)` уровня ниже. **`dstsize` обязателен:** `(1279 + 1) // 2 = 640`, удвоение даёт 1280 — без обрезки формы не сойдутся;
3. `gauss_pyr(x, n)` — список из **`n + 1`** уровней: исходный кадр и `n` уменьшенных;
4. `lap_pyr(x, n)` — список из `n + 1` уровня: `n` разностей `G[k] − pyr_up(G[k+1])` и вершина — последний уровень пирамиды Гаусса. **Считать во `float32`**;
5. `collapse(p)` — сборка от вершины: `pyrUp` до размера уровня ниже и прибавить.

> **Подсказка.** `cv2.filter2D` для сепарабельных проходов — можно; `cv2.pyrDown`/`pyrUp` — нельзя. В `lap_pyr` вход сначала привести к `float32`.

In [ ]:
K1 = np.array([1, 4, 6, 4, 1], np.float32) / 16.0


def pyr_down(x):
    """Своё уменьшение вдвое: [1 4 6 4 1] / 16 по каждой оси, затем каждый второй пиксель."""
    x = np.asarray(x)
    h, w = x.shape[:2]
    if x.dtype == np.uint8:
        wi = np.array([1, 4, 6, 4, 1], np.int32)
        pad = cv2.copyMakeBorder(x, 2, 2, 2, 2, cv2.BORDER_REFLECT_101).astype(np.int32)
        acc = np.zeros(((h + 1) // 2, (w + 1) // 2) + x.shape[2:], np.int32)
        for i, a in enumerate(wi):
            for j, b in enumerate(wi):
                acc += a * b * pad[i:i + h:2, j:j + w:2]
        return ((acc + 128) >> 8).astype(np.uint8)
    y = cv2.sepFilter2D(x, -1, K1, K1, borderType=cv2.BORDER_REFLECT_101)
    return y[::2, ::2]


def pyr_up(x, dstsize):
    """Удвоение до размера уровня ниже: нули между отсчётами, то же ядро, ×4. dstsize — (w, h)."""
    x = np.asarray(x)
    h, w = x.shape[:2]
    up = np.zeros((2 * h, 2 * w) + x.shape[2:], dtype=x.dtype)
    up[::2, ::2] = x
    y = cv2.sepFilter2D(up, -1, K1, K1, borderType=cv2.BORDER_REFLECT_101) * 4.0
    return y[:dstsize[1], :dstsize[0]]


def gauss_pyr(x, n):
    """Пирамида Гаусса: исходный кадр + n уменьшенных — всего n + 1 уровней."""
    p = [np.asarray(x)]
    for _ in range(n):
        p.append(pyr_down(p[-1]))
    return p


def lap_pyr(x, n):
    """Пирамида Лапласа: разности соседних уровней Гаусса + вершина. Только float32."""
    g = gauss_pyr(np.asarray(x, np.float32), n)
    p = []
    for k in range(n):
        p.append(g[k] - pyr_up(g[k + 1], (g[k].shape[1], g[k].shape[0])))
    p.append(g[-1])
    return p


def collapse(p):
    """Сборка от вершины: удвоить до размера уровня ниже и прибавить."""
    x = np.asarray(p[-1], np.float32)
    for k in range(len(p) - 2, -1, -1):
        x = pyr_up(x, (p[k].shape[1], p[k].shape[0])) + np.asarray(p[k], np.float32)
    return x


g1 = pyr_down(crop)
assert g1.shape == ((crop.shape[0] + 1) // 2, (crop.shape[1] + 1) // 2), "размер уровня — (H + 1) // 2 × (W + 1) // 2, как у cv2.pyrDown"
d_pd = np.abs(g1 - cv2.pyrDown(crop)).max()
assert d_pd < 1e-6, "своя pyr_down против cv2.pyrDown: %.3g — ядро [1,4,6,4,1] поделено на 16? достройка REFLECT_101?" % d_pd
gp = gauss_pyr(crop, 3)
assert len(gp) == 4, "gauss_pyr(x, 3) — это 4 уровня: исходный + три уменьшенных"
lp = lap_pyr(crop, 3)
assert all(l.dtype == np.float32 for l in lp), "уровни пирамиды Лапласа — только float32: в uint8 cv2.subtract обнулит отрицательные"
err = np.abs(collapse(lp) - np.float32(crop)).max()
assert err < 1e-4, "обратимость: ошибка %.3g — проверьте dstsize в pyr_up и порядок сборки (от вершины)" % err
print("TODO 3 ✔ · pyr_down против cv2: %.1e · обратно: %.1e · в L0 отрицательных %.0f%%" % (d_pd, err, 100 * (lp[0] < 0).mean()))
viz.grid({**{"G%d" % i: cv2.normalize(g, None, 0, 1, cv2.NORM_MINMAX) for i, g in enumerate(gp)},
          **{"L%d" % i: cv2.normalize(l, None, 0, 1, cv2.NORM_MINMAX) for i, l in enumerate(lp[:3])}}, cols=4, size=2.2)

### Почему пирамида Лапласа не живёт в `uint8` (слайд 40)

Уровень Лапласа — полосовой фильтр: детали могут быть и темнее фона. В `L0` около 49% значений отрицательные. `cv2.subtract` в `uint8` обнуляет их — и кадр восстанавливается на 18.9 дБ вместо обратимой сборки.

In [ ]:
gp_u = [np.uint8(cv2.normalize(g, None, 0, 255, cv2.NORM_MINMAX)) for g in gauss_pyr(crop, 3)]  # только для вида; разности — в uint8 как у «неаккуратной» реализации
lp_u = [cv2.subtract(a, cv2.resize(b, (a.shape[1], a.shape[0]))) for a, b in zip(gp_u, gp_u[1:])]  # cv2.subtract: −5 → 0
print("в L0 отрицательных значений %.0f%% — cv2.subtract превратил их в нули" % (100 * (lp[0] < 0).mean()))
print("float32: обратимость %.1e кодов · та же пирамида в стиле 'вычитание в байтах' теряет почти всё" % err)

### Вопрос, на который отвечаем вслух

`pyrUp` без `dstsize` вернёт для уровня 640 строк 1280 вместо 1279. Почему? И второй вопрос: сколько памяти занимает вся пирамида Гаусса из семи уровней относительно исходного кадра? (Слайд 39.)

---

## TODO 4 — склейка по полосам частот (≈ 9 минут)

Слайды 42–43. Идея Бёрта и Адельсона (1983): оба кадра — в пирамиды Лапласа, маску шва — в пирамиду Гаусса; смешать уровень с уровнем своей маской и собрать обратно. Маска на уровне `k` размыта на ±2ᵏ⁺¹ px от шва: мелкие детали смешиваются в узкой зоне, перепады яркости — в широкой. Каждая полоса — на ширине порядка своей длины волны: двоиться нечему, шву негде появиться.

Пара кадров ниже — модель слайда 37: правая половина сдвинута на 3 px и темнее на 24 кода. Линейка одной шириной эту задачу не решает: узкая — виден шов по яркости, широкая — двоятся детали.

Напишите `blend_bands(A, B, mask, n=5)` → кадр того же размера, `float64` в диапазоне 0–255:

1. привести `A`, `B` и `mask` к `float32` (маска — 0…1);
2. `lap_pyr(A, n)`, `lap_pyr(B, n)`, `gauss_pyr(mask, n)`;
3. `L[k] = m · LA[k] + (1 − m) · LB[k]` — маска уровня `k` со своим размером (`m[..., None]` для цветных кадров);
4. `collapse(L)` и **обязательно `np.clip(..., 0, 255)`** — значения после сборки выходят за диапазон.

> **Подсказка.** При `n = 0` результат должен совпадать с обычной линейной маской `A · m + B · (1 − m)` — это проверяет ассерт.

In [ ]:
# Панорама из того же фото: правая половина темнее на 24 кода и сдвинута на 3 px (модель слайда 37)
Hh, Ww = photo.shape[:2]; xs = Ww // 2
left_f = np.float32(photo)
right_f = np.float32(np.roll(photo, 3, axis=1)) - 24.0
panA = np.clip(left_f, 0, 255)
panB = np.clip(np.where(np.arange(Ww)[None, :] < xs, left_f, right_f), 0, 255)
mask_f = np.float32(np.arange(Ww)[None, :] < xs) * np.ones((Hh, 1), np.float32)   # 1 слева, 0 справа


def blend_bands(A, B, mask, n=5):
    """Склейка по полосам частот: уровни Лапласа смешиваются маской своего уровня → кадр в 0–255."""
    A, B, mask = np.float32(A), np.float32(B), np.float32(mask)
    LA, LB, GM = lap_pyr(A, n), lap_pyr(B, n), gauss_pyr(mask, n)
    mixed = []
    for a, b, m in zip(LA, LB, GM):
        if a.ndim == 3:
            m = m[..., None]
        mixed.append(m * a + (1.0 - m) * b)
    return np.clip(collapse(mixed), 0, 255).astype(np.float64)


bl = blend_bands(panA, panB, mask_f, 5)
assert bl.shape == panA.shape and bl.dtype == np.float64, "результат того же размера, float64 после клипа"
m0 = blend_bands(panA, panB, mask_f, 0)
linear = np.float32(panA) * mask_f + np.float32(panB) * (1 - mask_f)
assert np.abs(m0 - linear).max() < 0.5, "n = 0 должно совпадать с линейной маской: проверьте, что маска строится на том же числе уровней"
assert bl.max() <= 255.0, "значения вышли за 255: после сборки нужен np.clip (слайд 43: от −22 до 255)"
strip = slice(xs - 320, xs + 320)
assert np.abs(bl[:, strip] - panB[:, strip]).mean() < 3.0, "шва не видно: в полосе вокруг xs результат должен быть близок к собранной правой части"
print("TODO 4 ✔ · переход яркости вокруг шва, резкость — в ячейке ниже")
viz.grid({"шов в панораме": np.abs(np.float32(panA) - np.float32(panB))[::8, xs - 160:xs + 160] / 60,
          "склейка по полосам": (bl[::8, xs - 160:xs + 160] / 255.0)}, cols=2, size=4)

### Один шов, четыре склейки (слайд 43)

Срез — переход 1 px, шов виден. Линейка 40 px — резкость падает, шов остаётся. Линейка 400 px — шва нет, но двоятся контуры. Склейка по полосам — широкий переход для яркости и узкий для деталей одновременно.

In [ ]:
def linear_blend(w):
    x = np.arange(Ww, dtype=np.float32)
    m = np.clip(0.5 - (x - xs) / w, 0, 1)[None, :] * np.ones((Hh, 1), np.float32)
    mm = m[..., None] if panA.ndim == 3 else m
    return np.clip(np.float32(panA) * mm + np.float32(panB) * (1 - mm), 0, 255)


def seam_report(name, out):
    col = np.float64(out[:, xs - 200:xs + 200].mean(axis=(1, 2) if out.ndim == 3 else 1))
    width = float(np.count_nonzero(np.abs(np.diff(col)) > 0.5))                      # ширина перехода яркости, грубо
    band = out[:, xs - 64:xs + 64, 0] if out.ndim == 3 else out[:, xs - 64:xs + 64]
    sharp = float(np.abs(np.diff(np.float64(band), axis=0)).mean())  # энергия вертикальных градиентов у шва
    print("%-18s переход яркости %5.0f px · резкость у шва %6.2f" % (name, width, sharp))


seam_report("срез по шву", linear_blend(1))
seam_report("линейка 40 px", linear_blend(40))
seam_report("линейка 400 px", linear_blend(400))
seam_report("полосы, 5 уровней", bl)
# по слайду 43: срез 1 px · линейка 40 — 33 px, 84% · линейка 400 — 321 px, 45% · полосы, 6 ур. — 134 px, 98%

### Вопрос, на который отвечаем вслух

Два кадра склеивают по полосам, перекрытие 200 px. Сколько уровней пирамиды допустимо и что случится, если взять на два больше? (Слайд 44: зона смешивания ±2ⁿ⁺¹ px должна помещаться в половину перекрытия — иначе в склейку втекают чёрные поля.)

---

## Клиника: ДЗ 1 финал + ДЗ 2 «Сканер документов» (последние 20 минут)

**ДЗ 1 «Фотолаборатория»** — дедлайн вс 11.10, 23:59. Чек-лист перед сдачей: `pytest tests -q` — 11 тестов на синтетике; потом те же команды на кадрах курса и сверка с `sample_data/README.md` до последнего знака: хромакей на `ball-lamp.jpg` — recall 94.6 %, precision 99.9 %; выпрямление `poster-angle.jpg` — 99.1 %; поворот на 30° — 2302 × 2068, пустого холста 48.4 %. Расхождение больше допуска — ошибка в функции, а не «другая машина».

**ДЗ 2 «Сканер документов»** выдаётся на этой неделе. Что из сегодняшнего занятия туда идёт:

| Тема занятия 6 | Где в ДЗ 2 | Правило |
|----------------|------------|---------|
| спектр как диагностика | поиск растра/муара на сканах | растр виден в спектре как решётка точек (слайд 21) |
| алиасинг | уменьшение страниц | `INTER_AREA` или Гаусс σ ≈ 0.3 шага до ресайза: 21.9 → 37.2 дБ (слайды 26–27) |
| `uint8` и `np.clip` | все точечные операции | считать во `float32`, обратно через `np.clip` (слайд 49) |
| фазовая корреляция | выравнивание сканов (если есть в ТЗ) | `cv2.phaseCorrelate`, сдвиг за 10 мс (слайд 13) |

### `uint8` переполняется молча — повторение перед сдачей ДЗ 1 (слайд 49)

In [ ]:
photo_bgr = cv2.cvtColor(photo, cv2.COLOR_GRAY2BGR)
wrapped = photo + np.uint8(100)                                        # арифметика uint8 — по модулю 256
print("photo + 100 в uint8: %.1f%% пикселей стали темнее; 200 + 100 = %d"
      % (100 * (wrapped < photo).mean(), (np.array([200], np.uint8) + np.uint8(100))[0]))
gains = np.array([1.0, 1.1, 1.4])                                      # «баланс белого»
bad = (photo_bgr * gains).astype(np.uint8)                             # 280 → 24, предупреждения нет
good = np.clip(photo_bgr * gains, 0, 255).astype(np.uint8)
print("усиление (1.0, 1.1, 1.4) без np.clip: испорчено %.1f%% пикселей — в шаблоне ДЗ для этого as_uint8" % (100 * (bad != good).any(2).mean()))

### Алиасинг: уменьшение ×¼ — тема ДЗ 2 (слайды 26–27, демо 2.1)

In [ ]:
size = (480, 320)                                                      # ×¼ от 1920 × 1279
ref = cv2.resize(cv2.GaussianBlur(photo, (0, 0), 1.25), size, interpolation=cv2.INTER_LINEAR)  # эталон: фильтр до прореживания
ways = {"каждый 4-й пиксель": cv2.resize(photo, size, interpolation=cv2.INTER_NEAREST),
        "INTER_LINEAR": cv2.resize(photo, size, interpolation=cv2.INTER_LINEAR),
        "INTER_AREA": cv2.resize(photo, size, interpolation=cv2.INTER_AREA),
        "Гаусс σ 1.25 + LINEAR": cv2.resize(cv2.GaussianBlur(photo, (0, 0), 1.25), size, interpolation=cv2.INTER_LINEAR)}
for name, im in ways.items():
    print("%-22s %.1f дБ" % (name, metrics.psnr(ref, im)))
# по слайдам 26–27: без фильтра ложные узоры (21.9 дБ), с фильтром σ ≈ 0.3 шага — 37.2; правило σ ≈ 0.25–0.3 шага

### Вопросы клиники

Принесли вопрос по ДЗ 1 или ДЗ 2 — задавайте сейчас. Если вопросов нет, разбираем чек-лист ДЗ 1 и связь сегодняшней лекции с ДЗ 2. С занятия 5 ассистент принимает **показ ДЗ**: одно ДЗ за семестр, две минуты — запуск команды и объяснение одного места кода.

---

## ⭐ Звёздочка (по желанию, +0.5, до конца следующей недели)

**Поиск шаблона от грубого к точному** (слайд 45). Шаблон ищется на вершине пирамиды (`cv2.matchTemplate`, `TM_CCOEFF_NORMED`), координата удваивается и уточняется в окне ±2 px на каждом уровне вниз. Три уровня — 0.6 мс вместо 35, ответ тот же; пять уровней — шаблон на вершине 3 × 3 и промах на сотни пикселей.

Напишите `pyr_search(img, tpl, n=3)` — ячейка ниже сверит ошибку с полным поиском и измерит время для `n = 0, 2, 3, 5`. Сдаётся ноутбук с функцией, таблицей и одной фразой: почему на пяти уровнях промах.

In [ ]:
tpl = photo[560:656, 960:1056]                                          # шаблон 96 × 96 из кадра (координаты слайда 45)


def pyr_search(img, tpl, n=3):
    """Поиск от грубого к точному: вершина пирамиды, затем удвоение координаты и уточнение в окне ±2 px на каждом уровне."""
    raise NotImplementedError("⭐ звёздочка: пирамиды кадра и шаблона, matchTemplate на вершине, уточнение вниз")


def search_ref(img, tpl):
    return cv2.minMaxLoc(cv2.matchTemplate(img, tpl, cv2.TM_CCOEFF_NORMED))[3]


try:
    pyr_search(photo, tpl, 3)
except NotImplementedError:
    print("⭐ не заполнена — это необязательная часть; замеры появятся, когда функция будет написана")
else:
    x0, y0 = search_ref(photo, tpl)
    for n in (0, 2, 3, 5):
        x, y = (pyr_search(photo, tpl, n) if n else search_ref(photo, tpl))
        print("уровней %d: %6.2f мс · ошибка %3.0f px" % (n, ms(lambda: (pyr_search(photo, tpl, n) if n else search_ref(photo, tpl)), 3), np.hypot(x - x0, y - y0)))

## Зачёт за занятие

Ассистент ставит зачёт, если:

- [ ] ячейка проверки окружения прошла;
- [ ] **TODO 1–4 выполнены**, все ассерты проходят;
- [ ] вы можете объяснить **теорему о свёртке** и почему свёртка через спектр не зависит от размера ядра;
- [ ] на вопросы «почему у края свёртка через спектр расходится с `filter2D`», «как найти и погасить пик периодической помехи» и «зачем пирамида Лапласа считает во `float`, а не в `uint8`» есть внятный ответ.

2 % итоговой оценки, ставится в классе. Досылать ничего не нужно. **Занятие 7 — Quiz 2, блок II (L4–L6).**